# Databricks notebook source

## 11 · SQL de consumo da Gold
##**Fase 3 · Gold** | Consultas para manutenção, operação e análise. Uma consulta por célula.

In [0]:
CATALOG = "project_lakehouse_telemetria_ele"
GOLD = f"{CATALOG}.gold"

In [0]:
# 1. Top 10 do ranking com z por sinal e IC95% da taxa de picos
# O IC95% é do próprio ativo e não tem correção para múltiplos testes; a decisão de atenção usa o `z_limite`.

display(spark.sql(f"""
SELECT posicao, device_id, nivel, z_max, sinal_dominante, z_picos, z_tensao, z_sentinela,
       pct_picos, pct_picos_ic95_min, pct_picos_ic95_max
FROM {GOLD}.ranking_risco_ativos ORDER BY posicao LIMIT 10
"""))

In [0]:
# 2. Picos por dia dos 5 primeiros do ranking
# Com 21.000 leituras por ativo e ~0,3% de picos, o esperado é ~9 por dia: oscilações pequenas são ruído.

display(spark.sql(f"""
SELECT d.data_evento, d.device_id, d.qtd_picos, d.leituras
FROM {GOLD}.dispositivo_dia d
JOIN {GOLD}.ranking_risco_ativos r ON r.device_id = d.device_id AND r.posicao <= 5
ORDER BY d.device_id, d.data_evento
"""))

In [0]:
# 3. Horários com mais alarmes e mais picos

display(spark.sql(f"""
SELECT HOUR(hora_ts) AS hora_do_dia,
       SUM(qtd_alarmes) AS alarmes, SUM(qtd_picos) AS picos,
       ROUND(100*SUM(qtd_alarmes)/SUM(leituras),3) AS pct_alarme,
       ROUND(100*SUM(qtd_picos)/SUM(leituras),3) AS pct_picos
FROM {GOLD}.dispositivo_hora
GROUP BY HOUR(hora_ts)
ORDER BY alarmes DESC, picos DESC
"""))

In [0]:
# 4. Ativos acima do limite estatístico (pode retornar zero linhas, e isso é um resultado válido)

display(spark.sql(f"""
SELECT posicao, device_id, z_max, sinal_dominante, nivel
FROM {GOLD}.ranking_risco_ativos
WHERE nivel = 'atenção' ORDER BY posicao
"""))

In [0]:
# 5. Perfil horário da temperatura (da Fase 1, agora na Gold)

# COMMAND ----------

display(spark.sql(f"""
SELECT hora_do_dia, temp_media_todas, temp_media_filtrada, qtd_picos
FROM {GOLD}.perfil_horario ORDER BY hora_do_dia
"""))